# <font color='blue'>Data Science Academy</font>
# <font color='blue'>Processamento de Linguagem Natural com Transformers</font>

## <font color='blue'>Lab 5</font>
## <font color='blue'>Chatbot Personalizado com IA Generativa e LangChain</font>

![DSA](images/Lab5.png)

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark.
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [ ]:
!pip install -q transformers einops accelerate langchain bitsandbytes xformers

In [ ]:
# Imports
import torch
import langchain
import transformers
from transformers import AutoTokenizer, pipeline
from langchain import HuggingFacePipeline, PromptTemplate, LLMChain
from langchain.chains import SimpleSequentialChain
from langchain.prompts import ChatPromptTemplate
from IPython.display import display, Markdown

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

In [ ]:
# Checando a GPU
!nvidia-smi

## Definindo o LLM Para IA Generativa

Este Lab usa o modelo de linguagem **bertin-project/bertin-gpt-j-6B-alpaca** do HuggingFace para gerar respostas aprimoradas com base em uma sequência de texto inicial.

O principal objetivo é melhorar as respostas geradas pelo modelo **bertin-project/bertin-gpt-j-6B-alpaca** usando técnicas como Prompt Templates e Sequential Chains.

O **bertin-project/bertin-gpt-j-6B-alpaca** é um modelo de linguagem de inteligência artificial baseado no modelo GPT-J 6B. É uma versão em espanhol deste modelo que foi treinada para gerar texto em espanhol. O modelo foi treinado usando o conjunto de dados Alpaca, que é uma versão limpa do conjunto de dados Alpaca criado em Stanford.

https://huggingface.co/bertin-project/bertin-gpt-j-6B-alpaca

In [ ]:
# Repositório do modelo
modelo = "bertin-project/bertin-gpt-j-6B-alpaca"

In [ ]:
# Tokenizador
tokenizer = AutoTokenizer.from_pretrained(modelo)

In [ ]:
# Cria o pipeline
pipeline = pipeline("text-generation",

                    # Modelo que será usado para geração de texto
                    model = modelo,

                    # Tokenizer
                    tokenizer = tokenizer,

                    # Tipo dos dados para computação
                    torch_dtype = torch.bfloat16,

                    # Se devemos confiar no código remoto ao carregar o modelo.
                    # Isso só deve ser definido como True se você confiar na origem do modelo.
                    trust_remote_code = True,

                    # O dispositivo a ser usado para os cálculos do modelo.
                    device_map = "auto",

                    # O comprimento máximo do texto a ser gerado.
                    max_length = 512,

                    # Se deve usar a amostragem ao gerar texto.
                    do_sample = True,

                    # O número de candidatos a manter durante a amostragem.
                    top_k = 2,

                    # O número de sequências de texto a serem geradas.
                    num_return_sequences = 1,

                    # O ID do token de fim de sequência.
                    eos_token_id = tokenizer.eos_token_id)

A temperatura é um hiperparâmetro em muitos modelos de linguagem que controla a aleatoriedade da saída do modelo. Um valor de temperatura mais baixo (como 0.2) faz com que o modelo produza saídas mais determinísticas, enquanto um valor de temperatura mais alto faz com que o modelo produza saídas mais aleatórias.

In [ ]:
# HuggingFacePipeline
llm = HuggingFacePipeline(pipeline = pipeline, model_kwargs = {"temperature": 0.2})

### Testando o Modelo Sem o Uso de Templates (Direto com Prompts)

In [ ]:
def imprime_resposta(string):
    display(Markdown(string))

In [ ]:
imprime_resposta(llm("Escribe un poema sobre el sol"))

In [ ]:
imprime_resposta(llm("Puedes decirme qué es una computadora?"))

In [ ]:
imprime_resposta(llm("Me explicarias que es la inteligencia artificial?"))

### Testando o Modelo com Prompt Templates do LangChain

In [ ]:
#langchain.debug = True

> Exemplo 1

In [ ]:
# Template
template = """
  Eres un chatbot experto en psicologia y neurociencia.
  Responde a la siguiente pregunta con respuestas brillantes.
  Sólo tienes que escribir la respuesta, no más.
  Pregunta: {pregunta}
  Respuesta:
"""

In [ ]:
# Prompt
prompt = PromptTemplate(template = template, input_variables = ["pregunta"])
print(prompt)

In [ ]:
# Chain
llm_chain = LLMChain(prompt = prompt, llm = llm)

In [ ]:
# Questão
question = "Explicame que es la atencion?"

In [ ]:
# Executa a chain
exec_exemplo1 = llm_chain.run(question)

In [ ]:
# Print
imprime_resposta(exec_exemplo1)

> Exemplo 2

In [ ]:
# Template
template = """
  Eres un chatbot inteligente, experto en Python, conoces todo sobre el lenguaje.
  Solo tienes responder a la pregunta no más.
  Pregunta: {pregunta}
  Respuesta:
"""

In [ ]:
# Prompt
prompt = PromptTemplate(template = template, input_variables = ["pregunta"])
print(prompt)

In [ ]:
# Chain
llm_chain = LLMChain(prompt = prompt, llm = llm)

In [ ]:
# Questão
question = "Que es un diccionario?"

In [ ]:
# Executa a chain
exec_exemplo2 = llm_chain.run(question)

In [ ]:
imprime_resposta(exec_exemplo2)

In [ ]:
# Questão
question = "Cuales son los tipos de datos basicos en python?"

In [ ]:
imprime_resposta(llm_chain.run(question))

## Sequential Chains

In [ ]:
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "pregunta: Cual es el mejor nombre para describir una empresa que fabrica {producto}, solo dame un nombre inovador e interesante y solo responde con el nombre. respuesta:"
)

In [ ]:
# Chain 1
chain_one = LLMChain(llm = llm, prompt = first_prompt)

In [ ]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Escribe una descripcion creativa de la siguiente compañia: {company_name}"
)

In [ ]:
# chain 2
chain_two = LLMChain(llm = llm, prompt = second_prompt)

In [ ]:
# SimpleSequentialChain
overall_simple_chain = SimpleSequentialChain(chains = [chain_one, chain_two], verbose = True)

In [ ]:
product = "Burritos veganos"

In [ ]:
exec_chain = overall_simple_chain.run(product)

In [ ]:
imprime_resposta(exec_chain)

# Fim